# Modelo de Recomendação — SVD + Similaridade de Itens

Abordagem: fatoração de matrizes com TruncatedSVD (sklearn) para obter vetores latentes dos filmes, seguida de similaridade de cosseno entre esses vetores para recomendar filmes similares ao histórico do usuário.

## 1. Setup

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

PROC_DIR = Path("../data/processed")

# --- Parâmetros ajustáveis ---
N_COMPONENTS        = 50   # fatores latentes do SVD
MIN_USER_RATINGS    = 20   # usuários com menos avaliações são removidos
MIN_MOVIE_RATINGS   = 50   # filmes com menos avaliações são removidos
TOP_N               = 10   # filmes recomendados por usuário
TOP_K_SIMILAR       = 20   # vizinhos mais próximos por filme avaliado
RANDOM_STATE        = 42

## 2. Carga dos dados

In [24]:
ratings = pd.read_csv(PROC_DIR / "ratings_sample.csv", parse_dates=["date"])
movies  = pd.read_csv(PROC_DIR / "movies_clean.csv")

print(f"Ratings carregados : {len(ratings):,}")
print(f"Filmes carregados  : {len(movies):,}")
ratings.head()

Ratings carregados : 2,500,010
Filmes carregados  : 59,047


,userId,movieId,rating,date
0,99476,104374,3.5,2016-07-07 13:17:20
1,107979,2634,4.0,2001-07-01 17:15:28
2,155372,1614,3.0,2004-10-16 00:45:31
3,65225,7153,4.0,2008-01-26 21:17:55
4,79161,500,5.0,2017-03-07 19:36:03


## 3. Filtragem de usuários e filmes com poucos dados

Usuários e filmes com poucas avaliações geram vetores latentes pouco confiáveis.
Removê-los reduz ruído sem perder os padrões principais.

In [25]:
# Filtra usuários
user_counts  = ratings["userId"].value_counts()
valid_users  = user_counts[user_counts >= MIN_USER_RATINGS].index
ratings = ratings[ratings["userId"].isin(valid_users)]

# Filtra filmes
movie_counts = ratings["movieId"].value_counts()
valid_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index
ratings = ratings[ratings["movieId"].isin(valid_movies)]

print(f"Após filtragem:")
print(f"  Usuários : {ratings['userId'].nunique():,}")
print(f"  Filmes   : {ratings['movieId'].nunique():,}")
print(f"  Ratings  : {len(ratings):,}")

Após filtragem:
  Usuários : 34,188
  Filmes   : 4,932
  Ratings  : 1,478,440


## 4. Divisão treino / teste (split temporal)

Ordenamos por data e usamos os 80% mais antigos para treino e os 20% mais recentes para teste.
Isso simula o cenário real: o modelo só vê avaliações passadas para recomendar filmes futuros.

In [26]:
ratings_sorted = ratings.sort_values("date")
split_idx      = int(len(ratings_sorted) * 0.8)
train          = ratings_sorted.iloc[:split_idx]
test           = ratings_sorted.iloc[split_idx:]

print(f"Treino : {len(train):,} ratings  (até {train['date'].max().date()})")
print(f"Teste  : {len(test):,}  ratings  (a partir de {test['date'].min().date()})")

Treino : 1,182,752 ratings  (até 2016-07-10)
Teste  : 295,688  ratings  (a partir de 2016-07-10)


## 5. Matriz esparsa usuário × filme

Reindexamos `userId` e `movieId` para índices contínuos (0, 1, 2, ...) e construímos
uma `csr_matrix` do scipy — formato eficiente para matrizes com muitos zeros.

In [27]:
user_idx  = {int(uid): i for i, uid in enumerate(train["userId"].unique())}
movie_idx = {int(mid): i for i, mid in enumerate(train["movieId"].unique())}
idx_movie = {i: mid for mid, i in movie_idx.items()}

rows = train["userId"].map(user_idx).to_numpy(dtype=int)
cols = train["movieId"].map(movie_idx).to_numpy(dtype=int)
data = train["rating"].to_numpy(dtype=float)

train_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_idx), len(movie_idx)))

n_users, n_movies = len(user_idx), len(movie_idx)
print(f"Matriz: {n_users:,} usuários × {n_movies:,} filmes")
print(f"Esparsidade: {1 - train_matrix.nnz / (n_users * n_movies):.2%}")

Matriz: 28,250 usuários × 4,793 filmes
Esparsidade: 99.13%


## 6. SVD — aprendendo os fatores latentes

**O que o SVD faz:** comprime a matriz usuário × filme em duas matrizes menores:
- `user_factors` (usuário × N_COMPONENTS): perfil latente de cada usuário
- `item_factors` (filme × N_COMPONENTS): perfil latente de cada filme

Os N_COMPONENTS fatores representam conceitos como "gosta de animação", "prefere drama", etc.
Mesmo filmes com poucas avaliações ganham um vetor denso por generalização.

In [28]:
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
user_factors = svd.fit_transform(train_matrix)        # (n_users × N_COMPONENTS)
item_factors = svd.components_.T                       # (n_movies × N_COMPONENTS)

variancia_explicada = svd.explained_variance_ratio_.sum()
print(f"Variância explicada pelos {N_COMPONENTS} componentes: {variancia_explicada:.2%}")
print(f"item_factors shape: {item_factors.shape}")

Variância explicada pelos 50 componentes: 10.91%
item_factors shape: (4793, 50)


## 7. Similaridade de cosseno entre itens

Com os vetores latentes densos, calculamos a similaridade entre todos os pares de filmes.
Resultado: matriz (n_filmes × n_filmes) onde cada célula [i, j] vai de 0 a 1.

In [29]:
item_sim = cosine_similarity(item_factors)  # (n_movies × n_movies)
print(f"Matriz de similaridade: {item_sim.shape}")

# Exemplo: 5 filmes mais similares ao filme de índice 0
ex_idx   = 0
ex_movie = movies[movies["movieId"] == idx_movie[ex_idx]]["title"].values[0]
top5_idx = np.argsort(item_sim[ex_idx])[::-1][1:6]
top5_ids = [idx_movie[i] for i in top5_idx]

print(f"\nFilme base: {ex_movie}")
print("Filmes similares:")
for idx, mid in zip(top5_idx, top5_ids):
    title = movies[movies["movieId"] == mid]["title"].values[0]
    print(f"  {title}  (sim={item_sim[ex_idx][idx]:.3f})")

Matriz de similaridade: (4793, 4793)

Filme base: Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
Filmes similares:
  Death and the Maiden (1994)  (sim=0.376)
  Shining, The (1980)  (sim=0.373)
  Let Me In (2010)  (sim=0.343)
  Double Team (1997)  (sim=0.340)
  Labyrinth (1986)  (sim=0.330)


## 8. Função de recomendação

**Lógica:**
1. Busca os filmes que o usuário avaliou no treino (histórico)
2. Para cada filme do histórico, encontra os TOP_K_SIMILAR mais similares
3. Pondera a similaridade pela nota dada pelo usuário (filmes bem avaliados influenciam mais)
4. Remove filmes já assistidos e retorna os TOP_N com maior score

In [30]:
def recommend(user_id: int, top_n: int = TOP_N) -> pd.DataFrame:
    if user_id not in user_idx:
        return pd.DataFrame(columns=["title", "genres", "score"])

    u = user_idx[user_id]
    user_row = train_matrix[u]

    # Filmes já avaliados (índices e notas)
    rated_indices = user_row.indices
    rated_ratings = user_row.data

    if len(rated_indices) == 0:
        return pd.DataFrame(columns=["title", "genres", "score"])

    # Acumula scores: similaridade × nota do usuário
    scores = np.zeros(item_sim.shape[0])
    for item_i, rating in zip(rated_indices, rated_ratings):
        sim_row  = item_sim[item_i]
        top_k    = np.argsort(sim_row)[::-1][1 : TOP_K_SIMILAR + 1]
        scores[top_k] += sim_row[top_k] * rating

    # Remove filmes já vistos
    scores[rated_indices] = 0

    # Top N índices
    top_indices  = np.argsort(scores)[::-1][:top_n]
    top_movie_ids = [idx_movie[i] for i in top_indices]
    top_scores    = scores[top_indices]

    result = movies[movies["movieId"].isin(top_movie_ids)].copy()
    score_map = dict(zip(top_movie_ids, top_scores))
    result["score"] = result["movieId"].map(score_map)
    return result[["title", "genres", "score"]].sort_values("score", ascending=False).reset_index(drop=True)

## 9. Avaliação do modelo

In [31]:
test_filtered = test[
    test["userId"].isin(user_idx) & test["movieId"].isin(movie_idx)
]

u_idx = test_filtered["userId"].map(user_idx).to_numpy(dtype=int)
m_idx = test_filtered["movieId"].map(movie_idx).to_numpy(dtype=int)

y_true = test_filtered["rating"].to_numpy(dtype=float)
y_pred = np.array([user_factors[u] @ item_factors[m] for u, m in zip(u_idx, m_idx)])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = float(np.mean(np.abs(y_true - y_pred)))

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"\n(Baseline ingênuo — média global — teria RMSE ≈ {float(y_true.std()):.4f})")

RMSE : 3.4339
MAE  : 3.2834

(Baseline ingênuo — média global — teria RMSE ≈ 1.0087)


In [32]:
# Precision@10: proporção de filmes recomendados que o usuário realmente avaliou bem no teste
RELEVANCE_THRESHOLD = 4.0  # nota >= 4.0 é considerada relevante

test_relevant = (
    test_filtered[test_filtered["rating"] >= RELEVANCE_THRESHOLD]
    .groupby("userId")["movieId"]
    .apply(set)
)

precisions = []
sample_users = test_relevant.index[:200]  # avalia em 200 usuários para agilidade

for uid in tqdm(sample_users, desc="Precision@10"):
    rec = recommend(uid, top_n=TOP_N)
    if rec.empty:
        continue
    rec_ids    = set(movies[movies["title"].isin(rec["title"])]["movieId"])
    hits       = len(rec_ids & test_relevant[uid])
    precisions.append(hits / TOP_N)

print(f"Precision@{TOP_N} : {np.mean(precisions):.4f}")

Precision@10: 100%|██████████| 200/200 [00:01<00:00, 148.97it/s]

Precision@10 : 0.0045


## 10. Exemplo de recomendação

In [33]:
USER_ID = 1

# Histórico do usuário
historico = (
    train[train["userId"] == USER_ID]
    .merge(movies, on="movieId")
    .sort_values("rating", ascending=False)[["title", "genres", "rating"]]
    .head(5)
)
print(f"Top 5 filmes avaliados pelo usuário {USER_ID}:")
display(historico)

print(f"\nTop {TOP_N} recomendações para o usuário {USER_ID}:")
display(recommend(USER_ID, top_n=TOP_N))

Top 5 filmes avaliados pelo usuário 1:


,title,genres,rating



Top 10 recomendações para o usuário 1:


,title,genres,score
